# Evaluation of the Discussion Winner for LLaMA

# Is the Toxic Agent winning?

In [1]:
import pandas as pd
import json
from pathlib import Path
from scipy import stats
import sys
sys.path.append('../src')
from src.extract_starting_agent import get_agent_with_opening_statement, get_agent_who_starts

In [2]:
df_llama = pd.read_csv("../data/evaluation mad/convergence_of_480_toxic_random_discussions_topic_16.csv", sep =",")
dt_llama = df_llama[df_llama["reason_for_convergence"] != "moderator detected alignment"]

In [3]:
print(df_llama.groupby("which_agent_is_toxic")["reason_for_convergence"].value_counts(normalize=True))

which_agent_is_toxic     reason_for_convergence      
con has toxic behaviour  moderator detected alignment    0.403433
                         pro has been convinced          0.373391
                         con has been convinced          0.223176
pro has toxic behaviour  moderator detected alignment    0.384615
                         con has been convinced          0.352227
                         pro has been convinced          0.263158
Name: proportion, dtype: float64


In [4]:
dt_llama["toxic_agent"] = dt_llama.apply(lambda row: 0 if row["which_agent_is_toxic"] == "pro has toxic behaviour" else 1 , axis=1) # 0 for pro is toxic, 1 for con is toxic

/var/folders/v6/9c6hcq397n90s3n8znn3v7vh0000gn/T/ipykernel_25007/172366765.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dt_llama["toxic_agent"] = dt_llama.apply(lambda row: 0 if row["which_agent_is_toxic"] == "pro has toxic behaviour" else 1 , axis=1) # 0 for pro is toxic, 1 for con is toxic


### One-Sample T-Test: The Toxic Agent for CON is winning significantly more than 50% of the time.

In [6]:
# when pro is toxic, does pro win?
pro_toxic = (dt_llama[dt_llama["toxic_agent"] == 0]["reason_for_convergence"] == "con has been convinced").astype(int)
t_stat, p_value = stats.ttest_1samp(pro_toxic, popmean=0.5)
print(f"PRO toxic:")
print(f"  win rate: {pro_toxic.mean():.4f}")
print(f"  p-value: {p_value:.4f}")

# when con is toxic, does con win?
con_toxic = (dt_llama[dt_llama["toxic_agent"] == 1]["reason_for_convergence"] == "pro has been convinced").astype(int)
t_stat, p_value = stats.ttest_1samp(con_toxic, popmean=0.5)
print(f"CON toxic:")
print(f"  win rate: {con_toxic.mean():.4f}")
print(f"  p-value: {p_value:.4f}")

PRO toxic:
  win rate: 0.5724
  p-value: 0.0743
CON toxic:
  win rate: 0.6259
  p-value: 0.0027


### Two-Sample T-Test: Both PRO and CON toxic agents are winning significantly more than their non-toxic counterparts.
### PRO

In [7]:
# when pro is toxic (toxic_agent == 0), pro wins = con gets convinced
pro_toxic_wins = (dt_llama[dt_llama["toxic_agent"] == 0]["reason_for_convergence"] == "con has been convinced").astype(int)

# when con is toxic (toxic_agent == 1), pro is NOT toxic -> pro wins = con gets convinced
pro_nontoxic_wins = (dt_llama[dt_llama["toxic_agent"] == 1]["reason_for_convergence"] == "con has been convinced").astype(int)

t_stat, p_value = stats.ttest_ind(pro_toxic_wins, pro_nontoxic_wins)
print(f"PRO toxic win rate: {pro_toxic_wins.mean():.4f}")
print(f"PRO non-toxic win rate: {pro_nontoxic_wins.mean():.4f}")
print(f"p-value: {p_value:.4f}")

PRO toxic win rate: 0.5724
PRO non-toxic win rate: 0.3741
p-value: 0.0007


### CON

In [8]:
con_toxic_wins = (dt_llama[dt_llama["toxic_agent"] == 1]["reason_for_convergence"] == "pro has been convinced").astype(int)
con_nontoxic_wins = (dt_llama[dt_llama["toxic_agent"] == 0]["reason_for_convergence"] == "pro has been convinced").astype(int)

t_stat, p_value = stats.ttest_ind(con_toxic_wins, con_nontoxic_wins)
print(f"CON toxic win rate: {con_toxic_wins.mean():.4f}")
print(f"CON non-toxic win rate: {con_nontoxic_wins.mean():.4f}")
print(f"p-value: {p_value:.4f}")

CON toxic win rate: 0.6259
CON non-toxic win rate: 0.4276
p-value: 0.0007


### AVOVA: Toxicity level has no effect on the winning rate of the Toxic Agent

In [9]:
dt_llama = dt_llama.copy()
dt_llama["pro_wins"] = (dt_llama["reason_for_convergence"] == "con has been convinced").astype(int)
dt_llama["con_wins"] = (dt_llama["reason_for_convergence"] == "pro has been convinced").astype(int)

In [10]:
groups_pro = [group["pro_wins"].values for _, group in dt_llama.groupby("toxicity_level")]
f_stat, p_value = stats.f_oneway(*groups_pro)
print(f"PRO wins by toxicity:")
print(f"  F-statistic: {f_stat:.4f}, P-value: {p_value:.4f}")
print(dt_llama.groupby("toxicity_level")["pro_wins"].mean())

PRO wins by toxicity:
  F-statistic: 0.1070, P-value: 0.8986
toxicity_level
mild        0.494624
moderate    0.461538
no          0.478723
Name: pro_wins, dtype: float64


In [11]:
groups_con = [group["con_wins"].values for _, group in dt_llama.groupby("toxicity_level")]
f_stat, p_value = stats.f_oneway(*groups_con)
print(f"CON wins by toxicity:")
print(f"  F-statistic: {f_stat:.4f}, P-value: {p_value:.4f}")
print(dt_llama.groupby("toxicity_level")["con_wins"].mean())

CON wins by toxicity:
  F-statistic: 0.1070, P-value: 0.8986
toxicity_level
mild        0.505376
moderate    0.538462
no          0.521277
Name: con_wins, dtype: float64


### AVOVA to check if the toxicity level has an effect on the rounds till convergence (equal to the paper MAD 1)


In [12]:
groups = [group["rounds_to_convergence"].values for _, group in dt_llama.groupby("toxicity_level")]

f_stat, p_value = stats.f_oneway(*groups)
print(f"F-statistic: {f_stat:.4f}")
print(f"P-value: {p_value:.4f}")

# group means for overview
print(dt_llama.groupby("toxicity_level")["rounds_to_convergence"].mean())

F-statistic: 12.4452
P-value: 0.0000
toxicity_level
mild        11.376344
moderate    11.240385
no           9.531915
Name: rounds_to_convergence, dtype: float64


# Is the Agent winning that starts the discussion?

In [13]:
directory = Path('../data/toxic_and_baseline_random_llama')

In [14]:
dt_llama = dt_llama.copy()
dt_llama["opening_agent"] = dt_llama["Path"].apply(get_agent_with_opening_statement(directory))

In [15]:
dt_llama.groupby("opening_agent")["reason_for_convergence"].value_counts(normalize=True)

opening_agent  reason_for_convergence
con            con has been convinced    0.507463
               pro has been convinced    0.492537
pro            pro has been convinced    0.547771
               con has been convinced    0.452229
Name: proportion, dtype: float64

### T-Test: The Agent who has the first opening statement does not win more often.

In [16]:
con_winning = (dt_llama[dt_llama["opening_agent"] == "con"]["reason_for_convergence"] == "pro has been convinced").astype(int)
con_loosing = (dt_llama[dt_llama["opening_agent"] == "con"]["reason_for_convergence"] == "con has been convinced").astype(int)

t_stat, p_value = stats.ttest_ind(con_winning, con_loosing)
print(f"t-statistic: {t_stat:.4f}")
print(f"p-value: {p_value:.4f}")

t-statistic: -0.2435
p-value: 0.8078


In [17]:
pro_winning = (dt_llama[dt_llama["opening_agent"] == "pro"]["reason_for_convergence"] == "pro has been convinced").astype(int)
pro_loosing = (dt_llama[dt_llama["opening_agent"] == "pro"]["reason_for_convergence"] == "con has been convinced").astype(int)

t_stat, p_value = stats.ttest_ind(pro_winning, pro_loosing)
print(f"t-statistic: {t_stat:.4f}")
print(f"p-value: {p_value:.4f}")

t-statistic: 1.6954
p-value: 0.0910


## Extract the Agent who starts the Discussion

In [18]:
dt_llama["starting_agent"] = dt_llama["Path"].apply(get_agent_who_starts(directory))

In [19]:
dt_llama.groupby("starting_agent")["reason_for_convergence"].value_counts(normalize=True)

starting_agent  reason_for_convergence
con             pro has been convinced    0.718519
                con has been convinced    0.281481
pro             con has been convinced    0.647436
                pro has been convinced    0.352564
Name: proportion, dtype: float64

In [20]:
dt_llama["starting_agent"].value_counts(normalize=True)

starting_agent
pro    0.536082
con    0.463918
Name: proportion, dtype: float64

### T-Test: The Agent who starts the discussion does win more often.


In [21]:
con_winning = (dt_llama[dt_llama["starting_agent"] == "con"]["reason_for_convergence"] == "pro has been convinced").astype(int)
con_loosing = (dt_llama[dt_llama["starting_agent"] == "con"]["reason_for_convergence"] == "con has been convinced").astype(int)

t_stat, p_value = stats.ttest_ind(con_winning, con_loosing)
print(f"t-statistic: {t_stat:.4f}")
print(f"p-value: {p_value:.4f}")

t-statistic: 7.9545
p-value: 0.0000


In [22]:
pro_winning = (dt_llama[dt_llama["starting_agent"] == "pro"]["reason_for_convergence"] == "con has been convinced").astype(int)
pro_loosing = (dt_llama[dt_llama["starting_agent"] == "pro"]["reason_for_convergence"] == "pro has been convinced").astype(int)

t_stat, p_value = stats.ttest_ind(pro_winning, pro_loosing)
print(f"t-statistic: {t_stat:.4f}")
print(f"p-value: {p_value:.4f}")

t-statistic: 5.4333
p-value: 0.0000


### One-Sample T-Test

In [23]:
# when con starts, do they win more than 50%?
con_outcomes = (dt_llama[dt_llama["starting_agent"] == "con"]["reason_for_convergence"] == "pro has been convinced").astype(int)
t_stat, p_value = stats.ttest_1samp(con_outcomes, popmean=0.5)
print(f"CON starter:")
print(f"  win rate: {con_outcomes.mean():.4f}")
print(f"  t-statistic: {t_stat:.4f}")
print(f"  p-value: {p_value:.4f}")

# when pro starts, do they win more than 50%?
pro_outcomes = (dt_llama[dt_llama["starting_agent"] == "pro"]["reason_for_convergence"] == "con has been convinced").astype(int)
t_stat, p_value = stats.ttest_1samp(pro_outcomes, popmean=0.5)
print(f"PRO starter:")
print(f"  win rate: {pro_outcomes.mean():.4f}")
print(f"  t-statistic: {t_stat:.4f}")
print(f"  p-value: {p_value:.4f}")

CON starter:
  win rate: 0.7185
  t-statistic: 5.6247
  p-value: 0.0000
PRO starter:
  win rate: 0.6474
  t-statistic: 3.8419
  p-value: 0.0002


### Anova for starter winns and toxicity level
Hpothesis: The winning rate of the starter of the discussion is independent of the toxicity level of the toxic agent (Its equal for all toxicity levels).

In [ ]:
dt_llama["starter_wins"] = (
        ((dt_llama["starting_agent"] == "con") & (dt_llama["reason_for_convergence"] == "pro has been convinced")) |
        ((dt_llama["starting_agent"] == "pro") & (dt_llama["reason_for_convergence"] == "con has been convinced"))
).astype(int)

dt_llama.groupby(["toxicity_level", "starting_agent"])["starter_wins"].mean().unstack()

In [ ]:
dt_llama.groupby("toxicity_level")["starter_wins"].mean()

In [52]:
dt_llama["winning_agent"] = dt_llama.apply(lambda row: 0 if row["reason_for_convergence"] == "pro has been convinced" else 1 , axis=1) # 0 for pro wins, 1 for con wins

/var/folders/v6/9c6hcq397n90s3n8znn3v7vh0000gn/T/ipykernel_5669/1388325257.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dt["winning_agent"] = dt.apply(lambda row: 0 if row["reason_for_convergence"] == "pro has been convinced" else 1 , axis=1) # 0 for pro wins, 1 for con wins


In [44]:
groups = [group["starter_wins"].values for _, group in dt_llama.groupby("toxicity_level")]

f_stat, p_value = stats.f_oneway(*groups)
print(f"F-statistic: {f_stat:.4f}")
print(f"P-value: {p_value:.4f}")

# group means for overview
print(dt_llama.groupby("toxicity_level")["starter_wins"].mean())

F-statistic: 1.4132
P-value: 0.2451
toxicity_level
mild        0.634409
moderate    0.663462
no          0.744681
Name: starter_wins, dtype: float64
